> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTIin49w/5zM403525G-teHXho4SLHg/view?utm_content=DAGzTIin49w&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h29b78664e1)。


# 1. 环境配置

## 1.1 python 环境准备

In [2]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3 pymupdf==1.26.7 jq==1.10.0 unstructured==0.18.26 openpyxl==3.1.5 networkx==3.6.1 msoffcrypto-tool==5.4.2 docx2txt==0.9 python-pptx==1.0.2 markdown==3.10 bilibili-api-python==17.4.1 youtube-transcript-api==1.2.3 pytube==15.0.0

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [3]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 文档切分简介

RAG（Retrieval Augmented Generation）是一种结合了检索（Retrieval）和生成（Generation）的技术，旨在通过利用外部知识库来增强大型语言模型（LLMs）的性能。它通过检索与用户输入相关的信息片段，并结合这些信息来生成更准确、更丰富的回答。

整个 RAG 的流程分为两步：
- 制作数据库
- 提问 + 调用数据库

而在制作数据库部分也包括三个部分：
- 文档载入
- 文档切分
- 向量数据库存储

本节我们将重点演示关于数据库制作中的文档载入部分内容。

# 2. 文档切分

## 2.1 简介
在文档内容被制作成向量数据库之前，我们需要先收集相关的内容：
- 从PDF、数据库、URLs等不同来源读取内容（目前一般只读取文本的相关内容）；
- 然后统一转成Documents格式，作为后续“切分—嵌入—检索”的输入。



## 2.2 载入方式

一般情况下，我们会通过 loader.load() 进行载入，其处理流程为：
- 遍历所有文件 / 页面
- 全部读完
- 全部变成 Document
- 一次性返回 list[Document]

虽然这种方法简单好理解，但是有一个很大的问题是非常吃内存。

因此对于数据量特别大的场景，langchain 中提供了 .lazy_load() 的载入方式，其不会立刻读数据，而是用完一个，再读下一个，这样的话整体的内存消耗就会被大大降低了。但是未必所有方法都支持，因此下面演示的内容将基于 loader.load() 实现。

### 2.2.1 载入 .txt 格式文件
最基础的 loader 格式，只需要安装 langchain_community 即可：

In [4]:
# 最基础的 loader 格式，只需要安装 langchain_community 即可
from langchain_community.document_loaders import TextLoader
loader = TextLoader(file_path="./loaders_example/sample.txt", encoding="utf-8")
pages = loader.load()
print(pages) # 将文档里所有的内容都打印出来

[Document(metadata={'source': './loaders_example/sample.txt'}, page_content='这是一个用于测试 LangChain TextLoader 的 TXT 示例文件。')]


### 2.2.2 载入网址内容
首先需要安装 beautifulsoup4 库:

In [5]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()
print(docs[0].page_content[:500]) # 将网页里面前500个字符打印出来

USER_AGENT environment variable not set, consider setting it to identify your requests.










1. 引言 — 动手学深度学习 2.0.0 documentation

























1. 引言






search








      Quick search
      


code


Show Source








                  MXNet
              


                  PyTorch
              


                  Jupyter 记事本
              


                  课程
              


                  GitHub
              


                  English
              










Table Of Contents


前言
安装
符号


1. 引言
2. 预备知识
2.1. 数据操作
2.2. 数据预处理
2.3. 线性代数
2.4. 微积分
2.


### 2.2.3 载入 .csv 文件
只需要 langchain-community 即可使用：

In [6]:
# 只需要 langchain-community 即可 
from langchain_community.document_loaders.csv_loader import CSVLoader
loader = CSVLoader(file_path="./loaders_example/sample.csv", encoding="utf-8")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.csv', 'row': 0}, page_content='姓名: 张三\n成绩: 85'), Document(metadata={'source': './loaders_example/sample.csv', 'row': 1}, page_content='姓名: 李四\n成绩: 92')]


### 2.2.4 载入 .pdf 格式文件
首先需要安装 pymupdf 库才能使用：

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader(r"./loaders_example/sample.pdf")
pages = loader.load()
print(pages) 

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2025-12-20T18:00:21+08:00', 'source': './loaders_example/sample.pdf', 'file_path': './loaders_example/sample.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-20T18:00:21+08:00', 'trapped': '', 'modDate': "D:20251220180021+08'00'", 'creationDate': "D:20251220180021+08'00'", 'page': 0}, page_content='IIIIIIII LangChain PyMuPDFLoader I PDF IIIII')]


### 2.2.5 载入 .json 格式文件
首先需要安装 jq 库才可使用：

In [8]:
from langchain_community.document_loaders import JSONLoader
loader = JSONLoader(file_path="./loaders_example/sample.json", jq_schema=".", text_content=False) # "." 表示把整个 JSON 文件当做一个 Document
pages = loader.load()
print(pages) 

[Document(metadata={'source': 'D:\\课件\\4.检索增强搜索（RAG）应用实战\\4.1.文档载入\\loaders_example\\sample.json', 'seq_num': 1}, page_content='{"title": "JSON \\u793a\\u4f8b", "content": "\\u8fd9\\u662f\\u4e00\\u4e2a\\u7528\\u4e8e\\u6d4b\\u8bd5 JSONLoader \\u7684\\u793a\\u4f8b\\u6587\\u4ef6\\u3002"}')]


### 2.2.6 载入 .xlsx 格式文件
首先需要安装 unstructured 、msoffcrypto-tool、networkx、和 openpyxl 库：

In [9]:
from langchain_community.document_loaders import UnstructuredExcelLoader
loader = UnstructuredExcelLoader(r"./loaders_example/sample.xlsx")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.xlsx'}, page_content='姓名 成绩 张三 85 李四 92')]


### 2.2.7 载入 .docx 格式文件
首先需要安装 docx2txt 库：

In [10]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader(r"./loaders_example/sample.docx")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.docx'}, page_content='这是一个用于测试 LangChain Docx2txtLoader 的 DOCX 示例文件。')]


### 2.2.8 载入 .ppt 文件
首先需要安装 unstructured, python-magic, python-pptx 库：

In [11]:
from langchain_community.document_loaders import UnstructuredPowerPointLoader
loader = UnstructuredPowerPointLoader(r"./loaders_example/sample.pptx")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.pptx'}, page_content='LangChain PPTX 示例\n\nsample.pptx\n\n用于测试 PPTX 文档加载\n\n\n\nDocument Loader 演示\n\n本页用于演示\n\nPPTX → Document → RAG')]


### 2.2.9 载入 .html 格式文件
首先需要安装 beautifulsoup4 和 lxml 库：

In [12]:
from langchain_community.document_loaders import BSHTMLLoader
loader = BSHTMLLoader(r"./loaders_example/sample.html", open_encoding="utf-8")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.html', 'title': ''}, page_content='\n\nHTML 示例\n这是一个用于测试 BSHTMLLoader 的示例文件。\n\n\n')]


### 2.2.10 载入 .md 格式文件
首先需要安装 unstructured 和 markdown 库：

In [13]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader(r"./loaders_example/sample.md")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.md'}, page_content='Markdown 示例\n\n这是一个用于测试 UnstructuredMarkdownLoader 的示例文件。')]


### 2.2.11 载入 .ipynb 格式文件
不需要安装额外的库：

In [14]:
from langchain_community.document_loaders import NotebookLoader
loader = NotebookLoader(r"./loaders_example/sample.ipynb", include_outputs=True, max_output_length=20, remove_newline=True)
pages = loader.load()
print(pages) 

[Document(metadata={'source': 'loaders_example\\sample.ipynb'}, page_content='\'markdown\' cell: \'[\'# LangChain Loader 示例 Notebook\', \'\', \'这是一个用于测试 **.ipynb 文档加载** 的示例文件。\']\'\n\n\'code\' cell: \'["print(\'Hello LangChain Loader\')"]\'\n with output: \'[\'Hello LangChain Loader\']\'\n\n\'markdown\' cell: \'[\'## 小结\', \'Notebook 本质是 JSON 结构，\', \'适合用 Unstructured 或 JSON Loader 解析。\']\'\n\n')]


### 2.2.12 载入 .xml 格式文件
无需安装额外的库：

In [15]:
from langchain_community.document_loaders import UnstructuredXMLLoader
loader = UnstructuredXMLLoader(r"./loaders_example/sample.xml")
pages = loader.load()
print(pages) 

[Document(metadata={'source': './loaders_example/sample.xml'}, page_content='LangChain RAG 应用实战\n\n文档加载\n\n介绍 TextLoader、PDF Loader、Unstructured Loader\n\n文本切分\n\nRecursiveCharacterTextSplitter 的使用\n\n\n        XML 文件适合用于结构化数据示例。\n    ')]


### 2.2.13 载入 bilibili 字幕文件

比如我们想要获取 B 站上某个视频的字幕作为我们文档数据的话，LangChain 中也有相关支持的 BiliBiliLoader 可以使用（需要安装 bilibili-api-python）。

我们需要打开并登录 B 站。然后打开开发者模式（F12）并找到应用程序。然后我们可以在筛选器里搜索三部分内容，并把对应的值保留下来：
- SESSDATA = `"<your sessdata>"`
- BUVID3 = `"<your buvids>"`
- BILI_JCT = `"<your bili_jct>"`

然后我们就可以创建载入器（loader）来对有字幕的视频进行提取了（不添加无法正确返回内容）：


In [16]:
from langchain_community.document_loaders import BiliBiliLoader

loader = BiliBiliLoader(
  ["https://www.bilibili.com/video/BV1g84y1R7oE/"],
  sessdata = "<your sessdata>",
  buvid3 = "<your buvids>",
  bili_jct = "<your bili_jct>"
)

docs = loader.load()
print(docs)

[Document(metadata={'bvid': 'BV1g84y1R7oE', 'aid': 620074163, 'videos': 1, 'tid': 208, 'tid_v2': 2085, 'tname': '', 'tname_v2': '', 'copyright': 1, 'pic': 'http://i2.hdslb.com/bfs/archive/49c2a3612efe1caad023973ca56d979aabc30922.jpg', 'title': "Let's Learn English on a Hike! 🍂🚶🏼🎒 【英文字幕】", 'pubdate': 1698108310, 'ctime': 1698108310, 'desc': "Have you ever learned English in nature? In this English lesson I take you to a hiking trail near me and teach you all of the words and phrases you'll need to know to have a conversation about hiking.\n\nIn this free English class you'll learn words and phrases like: trail, path, treacherous, view, sign, observation deck, gear, hiking boots, dangerous animals, racoon, skunk, and more.\n\nYou'll even see a squirrel and hear me say the word, which is supposed to be one of the hardest English words to pronounce! Maybe hit pause at that point and try to shadow me saying the word (repeat it after me).\n\nI hope you enjoy this English lesson and I hope yo

c:\Users\76391\.conda\envs\langchain_course\Lib\site-packages\langchain_community\document_loaders\bilibili.py:131: UserWarning: No subtitles found for video: https://www.bilibili.com/video/BV1g84y1R7oE/. Returning empty transcript.
  warnings.warn(


### 2.2.14 载入 Youtube 字幕文件
类似的我们也可以获取 YouTube 上视频的字幕，这个相对没那么复杂，只需要使用 YoutubeLoader 并安装 pytube 和 youtube-transcript-api 库即可（需要外网才能够连接）：

In [17]:
from langchain_community.document_loaders import YoutubeLoader

loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=QsYGlZkevEg",
    add_video_info=True,
    language=["en", "id"],
    translation="en",
)

docs = loader.load()
print(docs)

HTTPError: HTTP Error 400: Bad Request